In [ ]:
# Atlas Load Time Test - Downloads parquet from CloudFront and measures load times

import nest_asyncio
nest_asyncio.apply()

import asyncio
from playwright.async_api import async_playwright
import statistics
import time
import os
import shutil
import pandas as pd
from pathlib import Path
import tempfile
import requests
from io import BytesIO

# ============================================================
# CONFIGURATION
# ============================================================

HTTP_AUTH_USER = "allsci"
HTTP_AUTH_PASS = "allscicorp!@#$"

# Supabase login credentials
SUPABASE_EMAIL = "rlalani@allsci.com"
SUPABASE_PASSWORD = "!!Casio1994$$"  # <-- UPDATE THIS

URLS = {
    "Production": {
        "url": "https://app.allsci.com/explore/clinical-trials",
        "login": "https://app.allsci.com/?login=true"
    },

}

# CloudFront URLs for actual deployed parquet data

CLOUDFRONT_URLS = {
    "Production": "https://app.allsci.com/cdn/visual/embeddings/clinical-trials-atlas/data/dataset.parquet",
    "Staging": "https://app-stg.allsci.com/cdn/visual/embeddings/clinical-trials-atlas/data/dataset.parquet",
}
# Cache directory for downloaded files
CACHE_DIR = Path(os.path.expanduser("~/.embedding_atlas_cache/test_deployment"))

REQUIRED_COLS = ['x', 'y', 'title', 'organization', 'date', 'year', 
                 'phase', 'trial_id', 'status', 'human_cluster_label', 
                 'source', 'allsci_url']

OPTIONAL_COLS = ['mesh_terms', 'last_update', 'first_submit_date', 
                 'status_verified_date', 'first_submit_qc_date', 'first_post_date', 
                 'last_post_date', 'start_date', 'primary_completion_date', 
                 'completion_date', 'min_age', 'max_age', 'sex', 
                 'healthy_volunteers', 'std_ages', 'intervention_names']

NUM_RUNS_NO_CACHE = 2  # Cold load tests (cache cleared before each)
NUM_RUNS_WITH_CACHE = 2  # Warm load tests (cache preserved between runs)
MAX_WAIT_SECONDS = 180

# ============================================================
# CLOUDFRONT DOWNLOAD
# ============================================================

def download_parquet_from_cloudfront(name, url, force=False):
    """Download parquet file from CloudFront, using cache if available."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    
    # Create a safe filename from the URL
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "").lower()
    cache_file = CACHE_DIR / f"{safe_name}.parquet"
    
    if cache_file.exists() and not force:
        size_mb = cache_file.stat().st_size / (1024 * 1024)
        print(f"  [cached] {cache_file.name} ({size_mb:.1f} MB)")
        return cache_file
    
    print(f"  Downloading from CloudFront...")
    print(f"     URL: {url}")
    
    try:
        # Use HTTP Basic Auth for QA environment
        auth = None
        if "d63915jpqe9qt.cloudfront.net" in url:
            auth = (HTTP_AUTH_USER, HTTP_AUTH_PASS)
        
        response = requests.get(url, auth=auth, stream=True, timeout=300)
        response.raise_for_status()
        
        # Get content length if available
        content_length = response.headers.get('content-length')
        if content_length:
            total_mb = int(content_length) / (1024 * 1024)
            print(f"     Size: {total_mb:.1f} MB")
        
        # Download to cache file
        with open(cache_file, 'wb') as f:
            downloaded = 0
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
        
        size_mb = cache_file.stat().st_size / (1024 * 1024)
        print(f"  [OK] Downloaded: {cache_file.name} ({size_mb:.1f} MB)")
        return cache_file
        
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] Download failed: {e}")
        return None

# ============================================================
# PARQUET ANALYSIS
# ============================================================

def analyze_parquet(name, url):
    """Download and analyze parquet file from CloudFront URL."""
    filepath = download_parquet_from_cloudfront(name, url)
    
    if filepath is None or not filepath.exists():
        print(f"[WARNING] Failed to download: {url}")
        return None
    
    print(f"\n{'='*60}")
    print(f"FILE: {name}: {filepath.name}")
    print(f"{'='*60}")
    
    orig_size_mb = filepath.stat().st_size / (1024 * 1024)
    print(f"Downloaded file size: {orig_size_mb:.1f} MB")
    
    df = pd.read_parquet(filepath)
    print(f"Rows: {len(df):,}")
    print(f"Total columns: {len(df.columns)}")
    
    cols_to_keep = REQUIRED_COLS.copy()
    for col in OPTIONAL_COLS:
        if col in df.columns:
            cols_to_keep.append(col)
    
    present_cols = [c for c in cols_to_keep if c in df.columns]
    extra_cols = [c for c in df.columns if c not in REQUIRED_COLS + OPTIONAL_COLS]
    
    print(f"Required: {len([c for c in REQUIRED_COLS if c in df.columns])}/{len(REQUIRED_COLS)}")
    print(f"Optional: {len([c for c in OPTIONAL_COLS if c in df.columns])}/{len(OPTIONAL_COLS)}")
    print(f"Extra (filtered): {len(extra_cols)}")
    
    df_filtered = df[present_cols].copy()
    with tempfile.NamedTemporaryFile(suffix='.parquet', delete=False) as f:
        temp_path = f.name
    df_filtered.to_parquet(temp_path)
    filtered_size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    os.unlink(temp_path)
    
    print(f"After filtering: {filtered_size_mb:.1f} MB")
    
    col_sizes = [(col, df_filtered[col].memory_usage(deep=True) / (1024 * 1024)) 
                 for col in df_filtered.columns]
    col_sizes.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n columns by size:")
    for col, size in col_sizes:
        print(f"   {size:6.1f} MB | {col}")
    
    return {
        "orig_size_mb": orig_size_mb,
        "filtered_size_mb": filtered_size_mb,
        "rows": len(df),
        "cols": len(present_cols)
    }

# ============================================================
# LOAD TIME TESTING
# ============================================================

async def clear_cache(context, page):
    await context.clear_cookies()
    try:
        client = await page.context.new_cdp_session(page)
        await client.send("Network.clearBrowserCache")
        await client.send("Network.setCacheDisabled", {"cacheDisabled": True})
        await client.detach()
    except:
        pass

async def auto_login(page, login_url, email, password):
    """Automatically login using the AllSci login form."""
    print(f"    Navigating to login...", flush=True)
    await page.goto(login_url, wait_until="domcontentloaded", timeout=60000)
    
    # Wait for the login form to appear
    await page.wait_for_selector('input[name="email"]', timeout=15000)
    await asyncio.sleep(1)
    
    # Fill email
    await page.locator('input[name="email"]').fill(email)
    print(f"    Email entered", flush=True)
    
    # Fill password
    await page.locator('input[name="password"]').fill(password)
    print(f"    Password entered", flush=True)
    
    # Click the Sign In button
    await page.locator('button[type="submit"]:has-text("Sign In")').click()
    print(f"    Sign In clicked...", flush=True)
    
    # Wait for login to complete
    try:
        await page.wait_for_url(lambda url: "login" not in url.lower(), timeout=30000)
        print(f"    [OK] Login successful!", flush=True)
    except:
        await asyncio.sleep(5)
        print(f"    [OK] Login appears complete", flush=True)
    
    await asyncio.sleep(2)

async def measure_load_time(page, url, track_resources=True):
    """Measure load time by waiting for clinical-trials-atlas search.worker.
    
    Also tracks individual resource load times for bottleneck analysis.
    """
    
    state = {'worker_loaded': False, 'load_time': None}
    resources = {
        'js': [],
        'wasm': [],
        'parquet': [],
        'extensions': [],
        'html': [],
        'other': []
    }
    timeline = []
    start_time = None
    
    def categorize_url(url):
        if 'dataset.parquet' in url:
            return 'parquet'
        elif '.duckdb_extension.wasm' in url:
            return 'extensions'
        elif '.wasm' in url:
            return 'wasm'
        elif '.js' in url:
            return 'js'
        elif '.html' in url or url.endswith('/'):
            return 'html'
        else:
            return 'other'
    
    def on_response(response):
        nonlocal start_time
        if start_time is None:
            return
            
        elapsed = time.time() - start_time
        url = response.url
        
        # Get content length from headers
        try:
            size = int(response.headers.get('content-length', 0))
        except:
            size = 0
        
        # Track search worker completion
        if "clinical-trials-atlas" in url and "search.worker" in url:
            state['worker_loaded'] = True
            state['load_time'] = elapsed
            timeline.append({'time': elapsed, 'event': 'Search worker ready', 'url': url})
            print(f"    Search worker loaded! ({elapsed:.1f}s)", flush=True)
        
        # Categorize and track all resources
        if track_resources:
            category = categorize_url(url)
            # Extract filename from URL
            filename = url.split('/')[-1].split('?')[0][:40]
            resources[category].append({
                'url': url,
                'filename': filename,
                'size': size,
                'time': elapsed
            })
            
            # Add significant resources to timeline
            if size > 100000 or category in ['parquet', 'wasm', 'extensions']:
                timeline.append({
                    'time': elapsed,
                    'event': f'{category}: {filename}',
                    'size': size
                })
    
    page.on("response", on_response)
    
    print("    Navigating...", flush=True)
    start_time = time.time()
    timeline.append({'time': 0, 'event': 'Navigation started'})
    
    await page.goto(url, wait_until="domcontentloaded", timeout=180000)
    timeline.append({'time': time.time() - start_time, 'event': 'DOM content loaded'})
    print("    Waiting for clinical-trials search worker...", flush=True)
    
    last_status = 0
    while True:
        elapsed = time.time() - start_time
        
        if elapsed > MAX_WAIT_SECONDS:
            print(f"    [TIMEOUT]", flush=True)
            break
        
        if state['worker_loaded']:
            print(f"    [OK] Ready!", flush=True)
            break
        
        if int(elapsed) >= last_status + 10:
            last_status = int(elapsed)
            print(f"    Waiting... ({int(elapsed)}s)", flush=True)
        
        await asyncio.sleep(0.5)
    
    load_time = time.time() - start_time
    page.remove_listener("response", on_response)
    
    return {
        "total_time": load_time,
        "resources": resources,
        "timeline": sorted(timeline, key=lambda x: x['time'])
    }

def print_resource_breakdown(result):
    """Print detailed breakdown of resource load times."""
    if 'resources' not in result:
        return
    
    resources = result['resources']
    
    print(f"\n    RESOURCE BREAKDOWN:")
    for category in ['js', 'wasm', 'parquet', 'extensions', 'html', 'other']:
        items = resources.get(category, [])
        if not items:
            continue
        
        total_size = sum(r['size'] for r in items)
        max_time = max(r['time'] for r in items) if items else 0
        min_time = min(r['time'] for r in items) if items else 0
        duration = max_time - min_time if len(items) > 1 else max_time
        
        size_mb = total_size / (1024 * 1024)
        speed = size_mb / duration if duration > 0 else 0
        
        print(f"      {category.upper():12} {len(items):3} files, {size_mb:6.1f} MB, loaded by {max_time:5.1f}s")
    
    # Print timeline
    if 'timeline' in result and result['timeline']:
        print(f"\n    TIMELINE:")
        for event in result['timeline'][:15]:  # Show first 15 events
            size_str = f" ({event.get('size', 0) / (1024*1024):.1f} MB)" if event.get('size', 0) > 0 else ""
            print(f"      {event['time']:5.1f}s - {event['event']}{size_str}")

async def run_load_tests():
    user_data_dir = os.path.expanduser("~/.playwright-allsci-fresh")
    
    if os.path.exists(user_data_dir):
        shutil.rmtree(user_data_dir)
    
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir,
            headless=False,
            viewport={"width": 1920, "height": 1080},
            http_credentials={"username": HTTP_AUTH_USER, "password": HTTP_AUTH_PASS}
        )
        
        page = context.pages[0] if context.pages else await context.new_page()
        results = {}
        
        for name, config in URLS.items():
            print(f"\n{'='*60}")
            print(f"LOGIN: {name}")
            print(f"{'='*60}")
            
            await clear_cache(context, page)
            
            # Try auto-login
            if SUPABASE_PASSWORD != "YOUR_PASSWORD_HERE":
                try:
                    await auto_login(page, config["login"], SUPABASE_EMAIL, SUPABASE_PASSWORD)
                except Exception as e:
                    print(f"    [WARNING] Auto-login failed: {e}")
                    input(f"\n>>> Manual login required. Press Enter when done...")
            else:
                await page.goto(config["login"], wait_until="domcontentloaded")
                input(f"\n>>> Login manually, then press Enter...")
            
            # ---- TEST WITHOUT CACHE (Cold Load) ----
            print(f"\n{'='*60}")
            print(f"Testing: {name} - WITHOUT CACHE (Cold Load)")
            print(f"{'='*60}")
            
            no_cache_results = []
            for i in range(NUM_RUNS_NO_CACHE):
                print(f"\nCold Run {i+1}/{NUM_RUNS_NO_CACHE}:", flush=True)
                print("    Clearing cache...", flush=True)
                await clear_cache(context, page)
                
                try:
                    result = await measure_load_time(page, config["url"])
                    no_cache_results.append(result)
                    print(f"  [OK] TOTAL: {result['total_time']:.1f}s")
                    # Show detailed breakdown for first cold run only
                    if i == 0:
                        print_resource_breakdown(result)
                except Exception as e:
                    print(f"  [ERROR] {e}")
            
            # ---- TEST WITH CACHE (Warm Load) ----
            print(f"\n{'='*60}")
            print(f"Testing: {name} - WITH CACHE (Warm Load)")
            print(f"{'='*60}")
            
            # First, do one load to warm the cache
            print(f"\nWarming cache...", flush=True)
            await clear_cache(context, page)
            try:
                await measure_load_time(page, config["url"])
                print("  [OK] Cache warmed")
            except:
                pass
            
            with_cache_results = []
            for i in range(NUM_RUNS_WITH_CACHE):
                print(f"\nWarm Run {i+1}/{NUM_RUNS_WITH_CACHE}:", flush=True)
                print("    Using cached data...", flush=True)
                # Don't clear cache - just reload the page
                
                try:
                    result = await measure_load_time(page, config["url"])
                    with_cache_results.append(result)
                    print(f"  [OK] TOTAL: {result['total_time']:.1f}s")
                except Exception as e:
                    print(f"  [ERROR] {e}")
            
            # Store both results
            if no_cache_results or with_cache_results:
                results[name] = {
                    "no_cache": {
                        "avg": statistics.mean([r["total_time"] for r in no_cache_results]) if no_cache_results else 0,
                        "times": [r["total_time"] for r in no_cache_results],
                        "resources": no_cache_results[0].get("resources") if no_cache_results else None,
                        "timeline": no_cache_results[0].get("timeline") if no_cache_results else None
                    },
                    "with_cache": {
                        "avg": statistics.mean([r["total_time"] for r in with_cache_results]) if with_cache_results else 0,
                        "times": [r["total_time"] for r in with_cache_results]
                    }
                }
        
        await context.close()
        return results

# ============================================================
# RUN EVERYTHING
# ============================================================

print("="*60)
print("STEP 1: DOWNLOAD & ANALYZE PARQUET FILES FROM CLOUDFRONT")
print("="*60)

parquet_info = {}
for name, url in CLOUDFRONT_URLS.items():
    info = analyze_parquet(name, url)
    if info:
        parquet_info[name] = info

print("\n" + "="*60)
print("STEP 2: LOAD TIME TESTS")
print("="*60)

load_results = asyncio.get_event_loop().run_until_complete(run_load_tests())

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

# Cold Load Results (No Cache)
print(f"\nCOLD LOAD (No Cache):")
print(f"{'Environment':<20} | {'Size (MB)':>10} | {'Load (s)':>10} | {'MB/s':>8}")
print(f"{'-'*55}")

for name in URLS.keys():
    if name in parquet_info and name in load_results:
        size = parquet_info[name]['filtered_size_mb']
        load = load_results[name]['no_cache']['avg']
        speed = size / load if load > 0 else 0
        print(f"{name:<20} | {size:>10.1f} | {load:>10.1f} | {speed:>8.1f}")

# Warm Load Results (With Cache)
print(f"\nWARM LOAD (With Cache):")
print(f"{'Environment':<20} | {'Size (MB)':>10} | {'Load (s)':>10} | {'MB/s':>8}")
print(f"{'-'*55}")

for name in URLS.keys():
    if name in parquet_info and name in load_results:
        size = parquet_info[name]['filtered_size_mb']
        load = load_results[name]['with_cache']['avg']
        speed = size / load if load > 0 else 0
        print(f"{name:<20} | {size:>10.1f} | {load:>10.1f} | {speed:>8.1f}")

# Cache Impact Analysis
print(f"\nCACHE IMPACT:")
print(f"{'Environment':<20} | {'Cold (s)':>10} | {'Warm (s)':>10} | {'Speedup':>10}")
print(f"{'-'*55}")

for name in URLS.keys():
    if name in load_results:
        cold = load_results[name]['no_cache']['avg']
        warm = load_results[name]['with_cache']['avg']
        speedup = (cold - warm) / cold * 100 if cold > 0 else 0
        print(f"{name:<20} | {cold:>10.1f} | {warm:>10.1f} | {speedup:>9.0f}%")

# Environment Comparison
if len(load_results) == 2:
    names = list(load_results.keys())
    prod_cold = load_results[names[0]]['no_cache']['avg']
    prod_warm = load_results[names[0]]['with_cache']['avg']
    stg_cold = load_results[names[1]]['no_cache']['avg']
    stg_warm = load_results[names[1]]['with_cache']['avg']
    prod_size = parquet_info.get(names[0], {}).get('filtered_size_mb', 0)
    stg_size = parquet_info.get(names[1], {}).get('filtered_size_mb', 0)
    
    print(f"\nENVIRONMENT COMPARISON:")
    print(f"   Cold Load: Prod {prod_cold:.1f}s vs STG {stg_cold:.1f}s (diff: {prod_cold - stg_cold:.1f}s)")
    print(f"   Warm Load: Prod {prod_warm:.1f}s vs STG {stg_warm:.1f}s (diff: {prod_warm - stg_warm:.1f}s)")
    if prod_size > stg_size:
        print(f"   Size diff: {prod_size - stg_size:.1f} MB extra data in Prod")

# Bottleneck Analysis
print(f"\n{'='*60}")
print("BOTTLENECK ANALYSIS (from first cold load)")
print(f"{'='*60}")

for name in URLS.keys():
    if name in load_results and load_results[name]['no_cache'].get('resources'):
        print(f"\n{name}:")
        resources = load_results[name]['no_cache']['resources']
        timeline = load_results[name]['no_cache'].get('timeline', [])
        
        # Calculate totals by category
        category_stats = []
        for category in ['js', 'wasm', 'parquet', 'extensions']:
            items = resources.get(category, [])
            if items:
                total_size = sum(r['size'] for r in items)
                max_time = max(r['time'] for r in items)
                category_stats.append({
                    'category': category.upper(),
                    'count': len(items),
                    'size_mb': total_size / (1024 * 1024),
                    'loaded_by': max_time
                })
        
        # Sort by load completion time
        category_stats.sort(key=lambda x: x['loaded_by'])
        
        print(f"  {'Category':<12} {'Files':>6} {'Size':>10} {'Loaded by':>12}")
        print(f"  {'-'*42}")
        for stat in category_stats:
            print(f"  {stat['category']:<12} {stat['count']:>6} {stat['size_mb']:>9.1f}MB {stat['loaded_by']:>11.1f}s")
        
        # Show bottleneck
        if category_stats:
            slowest = max(category_stats, key=lambda x: x['loaded_by'])
            print(f"\n  BOTTLENECK: {slowest['category']} ({slowest['size_mb']:.1f} MB) - completed at {slowest['loaded_by']:.1f}s")

STEP 1: DOWNLOAD & ANALYZE PARQUET FILES FROM CLOUDFRONT
  [cached] production.parquet (52.5 MB)

FILE: Production: production.parquet
Downloaded file size: 52.5 MB
Rows: 555,279
Total columns: 15
Required: 12/12
Optional: 1/16
Extra (filtered): 2
After filtering: 50.4 MB

 columns by size:
     78.1 MB | title
     68.8 MB | allsci_url
     54.6 MB | mesh_terms
     50.1 MB | human_cluster_label
     46.0 MB | organization
     39.7 MB | source
     37.1 MB | status
     36.3 MB | phase
     36.0 MB | trial_id
     35.5 MB | date
     32.3 MB | year
      2.1 MB | x
      2.1 MB | y
  [cached] staging.parquet (37.7 MB)

FILE: Staging: staging.parquet
Downloaded file size: 37.7 MB
Rows: 555,279
Total columns: 20
Required: 4/12
Optional: 0/16
Extra (filtered): 16
After filtering: 37.9 MB

 columns by size:
     78.1 MB | title
     36.0 MB | trial_id
      2.1 MB | x
      2.1 MB | y

STEP 2: LOAD TIME TESTS

LOGIN: Production
    Navigating to login...
    Email entered
    Password en

In [ ]:
import pandas as pd
import requests
from io import BytesIO

# Download parquet from CloudFront
url = "https://app-stg.allsci.com/cdn/visual/embeddings/clinical-trials-atlas/data/dataset.parquet"
resp = requests.get(url, timeout=300)
df = pd.read_parquet(BytesIO(resp.content))

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

# Check if min_age and max_age exist
age_cols = [c for c in df.columns if 'age' in c.lower()]
print(f"Age columns found: {age_cols}")

# Analyze min_age and max_age distributions
for col in ['Target Min Age', 'Target Max Age']:
    if col in df.columns:
        print(f"\n=== {col} ===")
        print(df[col].describe())
        print(f"\nSample values:\n{df[col].dropna().head(20).tolist()}")
        
        # Find high values (likely in days if > 120)
        high_vals = df[df[col].notna() & (df[col] > 120)][[col, 'title', 'trial_id']].head(20)
        print(f"\nHigh values (> 120):\n{high_vals}")

Total rows: 555,279
Columns: ['x', 'y', 'title', 'trial_id', 'Sponsor Organization', 'First Post Date', 'Phase', 'Status', 'AllSci Cluster Label', 'Source', 'AllSci URL', 'sponsor_org_allsci_id', 'sponsor_org_aliases', 'MeSH Terms', 'First Submitted Date', 'Target Min Age', 'Target Max Age', 'Eligible Genders', 'Healthy Volunteers', 'First Submitted Year']


In [ ]:
# Check value distribution to detect days vs years
for col in ['Target Min Age', 'Target Max Age']:
    if col in df.columns:
        vals = df[col].dropna()
        
        # Count by range
        print(f"\n=== {col} distribution ===")
        print(f"  0-18:    {len(vals[(vals >= 0) & (vals <= 18)]):,}")
        print(f"  19-120:  {len(vals[(vals > 18) & (vals <= 120)]):,}")
        print(f"  121-365: {len(vals[(vals > 120) & (vals <= 365)]):,} (likely days)")
        print(f"  366+:    {len(vals[vals > 365]):,} (likely days)")
        
        # Top 10 highest values
        print(f"\n  Top 10 highest values:")
        for v in vals.nlargest(10).values:
            years = v / 365 if v > 365 else v
            print(f"    {v} {'(' + f'{years:.1f} years)' if v > 365 else ''}")


=== Target Min Age distribution ===
  0-18:    415,969
  19-120:  102,513
  121-365: 41 (likely days)
  366+:    5 (likely days)

  Top 10 highest values:
    730.0 (2.0 years)
    510.0 (1.4 years)
    427.0 (1.2 years)
    427.0 (1.2 years)
    366.0 (1.0 years)
    365.0 
    365.0 
    365.0 
    365.0 
    365.0 

=== Target Max Age distribution ===
  0-18:    37,208
  19-120:  256,668
  121-365: 701 (likely days)
  366+:    15 (likely days)

  Top 10 highest values:
    6569.0 (18.0 years)
    4383.0 (12.0 years)
    2190.0 (6.0 years)
    2189.0 (6.0 years)
    1824.0 (5.0 years)
    1095.0 (3.0 years)
    915.0 (2.5 years)
    730.0 (2.0 years)
    578.0 (1.6 years)
    577.0 (1.6 years)


In [18]:
# Check value distribution and get NCT IDs for outliers
for col in ['Target Min Age', 'Target Max Age']:
    if col in df.columns:
        vals = df[col].dropna()
        
        # Count by range
        print(f"\n=== {col} distribution ===")
        print(f"  0-18:    {len(vals[(vals >= 0) & (vals <= 18)]):,}")
        print(f"  19-120:  {len(vals[(vals > 18) & (vals <= 120)]):,}")
        print(f"  121-365: {len(vals[(vals > 120) & (vals <= 365)]):,} (likely days)")
        print(f"  366+:    {len(vals[vals > 365]):,} (likely days)")
        
        # Top 10 highest values with NCT IDs
        print(f"\n  Top 10 highest values:")
        top10 = df[df[col].notna()].nlargest(10, col)[['trial_id', col]]
        for _, row in top10.iterrows():
            v = row[col]
            nct = row['trial_id']
            years = v / 365 if v > 365 else v
            suffix = f' ({years:.1f} years)' if v > 365 else ''
            print(f"    {nct}: {v}{suffix}")


=== Target Min Age distribution ===
  0-18:    415,969
  19-120:  102,513
  121-365: 41 (likely days)
  366+:    5 (likely days)

  Top 10 highest values:
    NCT02193022: 730.0 (2.0 years)
    NCT00303316: 510.0 (1.4 years)
    NCT00136604: 427.0 (1.2 years)
    NCT00228917: 427.0 (1.2 years)
    NCT06124157: 366.0 (1.0 years)
    NCT03015844: 365.0
    NCT01226953: 365.0
    NCT00847145: 365.0
    NCT01574274: 365.0
    NCT03126916: 365.0

=== Target Max Age distribution ===
  0-18:    37,208
  19-120:  256,668
  121-365: 701 (likely days)
  366+:    15 (likely days)

  Top 10 highest values:
    NCT02193022: 6569.0 (18.0 years)
    NCT07134088: 4383.0 (12.0 years)
    NCT01290029: 2190.0 (6.0 years)
    NCT01439867: 2189.0 (6.0 years)
    NCT01733862: 1824.0 (5.0 years)
    NCT03561168: 1095.0 (3.0 years)
    NCT05453630: 915.0 (2.5 years)
    NCT00136604: 730.0 (2.0 years)
    NCT00303316: 578.0 (1.6 years)
    NCT00228917: 577.0 (1.6 years)
